# Baseline Training — All Models
Trains each baseline model sequentially, saves checkpoints, metrics, and figures per model.

In [ ]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

PROJECT_ROOT = Path(os.path.abspath('')).parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

# --- Reproducibility & run mode (added by full-scale fix) ---
import random as _rnd
SEED = 42
_rnd.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# No GPU  -> quick verification mode (tiny balanced subset, 1 epoch, 224px)
# GPU     -> full paper protocol (full manifest, 20 epochs, 384px)
SMOKE_TEST = not torch.cuda.is_available()
print(f'Device: {device} | SMOKE_TEST: {SMOKE_TEST}')

In [ ]:
# Cell 2: Config (FIXED: prepared manifest, DGX-aware settings, smoke mode)
from src.dataset import AIDetectionDataset, ImageTransform, create_split_dataloaders
from src.config import Config
import src
PROJECT_ROOT = Path(src.__file__).resolve().parent.parent.parent
os.chdir(PROJECT_ROOT)

cfg = Config()
cfg.training.model_variant = 'base'
cfg.training.epochs = 1 if SMOKE_TEST else 20
cfg.training.image_size = 224 if SMOKE_TEST else 384
cfg.training.batch_size = 8 if SMOKE_TEST else 64
cfg.training.num_workers = 0 if SMOKE_TEST else 8
cfg.training.mixed_precision = torch.cuda.is_available()
cfg.dataset.val_split = 0.10
cfg.dataset.test_split = 0.10

# Prefer the confound-fixed manifest (Places365 cap + deepfake real frames)
_manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'train_manifest.csv'
if _manifest.exists():
    cfg.dataset.metadata_paths = [str(_manifest)]
    print(f'Using prepared manifest: {_manifest.name}')
else:
    print('WARNING: train_manifest.csv not found - falling back to clean_metadata.csv')
    print('         Before the full-scale run, execute:')
    print('         python dataset/scripts/prepare_training_manifest.py')

print(f'Variant={cfg.training.model_variant} epochs={cfg.training.epochs} '
      f'size={cfg.training.image_size} batch={cfg.training.batch_size}')

In [ ]:
# Cell 3: Shared stratified Train/Val/Test split
# Saved split indices guarantee every notebook (MFFT variants, baselines,
# ablations) trains and evaluates on the IDENTICAL split.
MAX_SAMPLES = 600 if SMOKE_TEST else None
_split_name = 'split_indices_smoke.json' if SMOKE_TEST else 'split_indices.json'
train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=cfg.dataset.metadata_paths,
    batch_size=cfg.training.batch_size,
    num_workers=cfg.training.num_workers,
    size=cfg.training.image_size,
    val_split=cfg.dataset.val_split,
    test_split=cfg.dataset.test_split,
    seed=SEED,
    use_weighted_sampler=True,
    split_index_path=str(PROJECT_ROOT / 'dataset' / 'metadata' / _split_name),
    max_samples=MAX_SAMPLES,
)
train_dataset = train_loader.dataset
val_dataset = val_loader.dataset
test_dataset = test_loader.dataset
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

# Compatibility view for downstream cells (figures/tables) that reference
# `full_dataset`: the union of the three splits.
class _FullView:
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)

full_dataset = _FullView(train_dataset.samples + val_dataset.samples + test_dataset.samples)
print(f'full_dataset view: {len(full_dataset)} samples')

In [ ]:
# Cell 4: Define All Baselines
from src.baselines import (
    SimpleCNN, LightViT, count_parameters,
    resnet18, resnet50, efficientnet_b0, vit_b_16, swin_t,
    CLIPBaseline, FreqDetect, deit_small,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}\n')

BASELINE_REGISTRY = {
    'SimpleCNN':      lambda: SimpleCNN(),
    'LightViT':       lambda: LightViT(img_size=cfg.training.image_size, depth=4, num_heads=4, embed_dim=192),
    'ResNet-18':      lambda: resnet18(),
    'ResNet-50':      lambda: resnet50(),
    'EfficientNet-B0': lambda: efficientnet_b0(),
    'ViT-B/16':       lambda: vit_b_16(img_size=cfg.training.image_size),
    'Swin-T':         lambda: swin_t(),
    'DeiT-S':         lambda: deit_small(img_size=cfg.training.image_size),
    'CLIP':           lambda: CLIPBaseline(img_size=cfg.training.image_size),
    'FreqDetect':     lambda: FreqDetect(img_size=cfg.training.image_size),
}

x = torch.randn(2, 3, cfg.training.image_size, cfg.training.image_size)
print(f"{'Model':<20} {'Params':>10} {'Output':>10}")
print('-' * 42)
for name, fn in BASELINE_REGISTRY.items():
    m = fn()
    p = count_parameters(m)
    o = list(m(x).shape)
    print(f'{name:<20} {p:>10,}  {str(o):>10}')

# ── Select which baselines to train ──
# Set MODELS_TO_TRAIN = list(BASELINE_REGISTRY.keys()) for all, or pick a subset:
MODELS_TO_TRAIN = list(BASELINE_REGISTRY.keys())
# e.g. MODELS_TO_TRAIN = ['ResNet-50', 'EfficientNet-B0']
print(f'\nWill train: {MODELS_TO_TRAIN}')

if SMOKE_TEST:
    MODELS_TO_TRAIN = ['SimpleCNN', 'ResNet-18', 'FreqDetect']
    print(f'SMOKE_TEST: training only {MODELS_TO_TRAIN}')

In [ ]:
# Cell 5: Training Loop for All Baselines
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix

NUM_EPOCHS = cfg.training.epochs
AMP = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)

for model_name in MODELS_TO_TRAIN:
    print('\n' + '='*70)
    print(f'Training {model_name}...')
    print('='*70)

    model = BASELINE_REGISTRY[model_name]().to(device)
    print(f'Parameters: {count_parameters(model):,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

    ckpt_dir = PROJECT_ROOT / 'model' / 'checkpoints' / f'{model_name.lower().replace("/", "_").replace("-", "_")}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc = 0
    best_epoch = -1

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f'{model_name} Epoch {epoch+1}/{NUM_EPOCHS}')
        for images, labels in pbar:
            try:
                images, labels = images.to(device), labels.to(device)
                with torch.amp.autocast('cuda', enabled=AMP):
                    logits = model(images)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            except Exception as e:
                print(f"  Warning: skipping bad batch: {e}")
                optimizer.zero_grad()
                continue

            total_loss += loss.item()
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}',
                             'acc': f'{correct/total*100:.2f}%',
                             'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        train_acc = correct / total * 100
        history["train_acc"].append(train_acc)
        history["train_loss"].append(total_loss / len(train_loader))

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                try:
                    images, labels = images.to(device), labels.to(device)
                    logits = model(images)
                    loss = criterion(logits, labels)
                    val_loss += loss.item()
                    preds = logits.argmax(dim=-1)
                    val_correct += (preds == labels).sum().item()
                    val_total += labels.size(0)
                except Exception as e:
                    print(f"  Warning: bad val batch: {e}")
                    continue

        val_acc = val_correct / val_total * 100
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss / len(val_loader))
        print(f'  train={train_acc:.2f}%, val={val_acc:.2f}%')

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_dir / 'best.pt')
            print(f'  * Saved best ({best_acc:.2f}%)')

    print(f'\n{model_name} done. Best val acc: {best_acc:.2f}% at epoch {best_epoch}')

    # Save final model
    torch.save(model.state_dict(), ckpt_dir / 'final.pt')

    # Save history
    results_dir = PROJECT_ROOT / 'paper' / 'result' / 'full_scale' / model_name.lower().replace('/', '_').replace('-', '_')
    results_dir.mkdir(parents=True, exist_ok=True)
    with open(results_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Evaluate on test set
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            try:
                images = images.to(device)
                logits = model(images)
                probs = F.softmax(logits, dim=-1)
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs[:, 1].cpu().numpy())
            except Exception as e:
                print(f"  Warning: bad test batch: {e}")
                continue

    y_true = np.array(all_labels)
    y_score = np.array(all_probs)
    y_pred = (y_score >= 0.5).astype(int)

    acc = (y_pred == y_true).mean() * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100
    auc = roc_auc_score(y_true, y_score)

    metrics = {
        'model': model_name,
        'params': count_parameters(model),
        'best_val_acc': round(best_acc, 2),
        'best_epoch': best_epoch,
        'test_accuracy': round(acc, 2),
        'test_precision': round(prec, 2),
        'test_recall': round(rec, 2),
        'test_f1': round(f1, 2),
        'test_auc': round(auc, 4),
    }
    with open(results_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f'Test: acc={acc:.2f}%, prec={prec:.2f}%, rec={rec:.2f}%, f1={f1:.2f}%, auc={auc:.4f}')
    print(f'Results saved to {results_dir}/')

In [ ]:
# Cell 6: Summary Table of All Baselines
all_metrics = []
for model_name in MODELS_TO_TRAIN:
    results_dir = PROJECT_ROOT / 'paper' / 'result' / 'full_scale' / model_name.lower().replace('/', '_').replace('-', '_')
    metrics_file = results_dir / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            all_metrics.append(json.load(f))

if all_metrics:
    df = pd.DataFrame(all_metrics).set_index('model')
    print('\n=== BASELINE COMPARISON ===')
    print(df.to_string())
    summary_path = PROJECT_ROOT / 'paper' / 'result' / 'full_scale' / 'baseline_summary.csv'
    df.to_csv(summary_path)
    print(f'\nSummary saved to {summary_path}')